# 31 · The Design Matrix — Features & Labels for the Ranker

**Purpose.** Assemble the single table the XGBoost ranker trains on. In tensor
language: cut the customer×item×time tensor at the three training planes
(days 450, 510, 570), contract the past into feature columns for every
(household, product) cell in the frozen candidate pool, binarize the 30-day
future slice into the label — then stack the three cuts. Repeat once at day
600 for validation. Outputs are **files, not kernel memory**:
`data/processed/ranker_train.parquet` and `ranker_valid.parquet`.

## Feature set (column → source → why)

| Feature | Source | Justified by |
|---|---|---|
| times_bought, days_since_last, due_ness | `household_product_snapshot(t)` | strongest signals (EDA §1, §3; due-ness curve) |
| baskets_30/90/365d, spend_30/365d, tenure_days, avg_basket_value | `customer_snapshot(t)` | customer activity level & habit |
| item_baskets_365d, item_households_365d, item_avg_price | computed as-of from `stg_transactions` | popularity base rate (mean-normalization role) |
| median_gap_days, gap_source | `mart_product_cycles` (rebuilt ≤ day 450) | product clock |
| in_buy_again, in_popularity, in_als_new | pool membership | source flags — which generator vouches for this pair |
| als_affinity | per-snapshot ALS fit, $p_u^\top q_i$ | learned-embedding signal (the course's whole model as one column) |
| **label** | `purchase_labels(t, 30)` | bought in the 30-day future slice |

## Leakage rules (all three enforced in this notebook)

1. **ALS is refit per snapshot** — an embedding trained through day 600 knows
   things day 450 must not.
2. **Item popularity is computed per snapshot** (trailing 365d) — never from
   the full-period `mart_item_stats` (EDA-only).
3. **Cycles come from day ≤ 450 only** (rebuilt in step 0) — legal for all planes.

## Missing-value policy

Candidates arriving from popularity/ALS that a household never bought have
NaN interaction features (times_bought → 0 is a fact; days_since_last and
due_ness stay **NaN — deliberately**). XGBoost learns a default direction per
split, so missingness itself is informative: "no purchase history with this
item" is signal, not noise. (Same principle as the DSCR non-reporting story.)

## Scale expectation

~1.1–1.4M rows per plane × 3 planes ≈ 3.5–4M training rows, ~20 columns;
positives ≈ 4–5%. Build time ~10–20 min (four ALS fits + four pool queries).

In [2]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import scipy.sparse as sp
from pathlib import Path

from implicit.als import AlternatingLeastSquares

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parents[1]

CFG = yaml.safe_load((ROOT / "configs" / "base.yaml").read_text())
con = duckdb.connect((ROOT / "db" / "retail.duckdb").as_posix(), read_only=True)

def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()

TRAIN_DAYS = CFG["snapshots"]["train"]          # [450, 510, 570]
VALID_DAY  = CFG["snapshots"]["valid"]          # 600
HORIZON    = CFG["label_horizon_days"]          # 30
BUDGETS    = {k: (10**9 if v == -1 else v) for k, v in CFG["candidates"].items()}
OUT_DIR    = ROOT / "data" / "processed"

print("train planes:", TRAIN_DAYS, "| valid:", VALID_DAY, "| horizon:", HORIZON)
print("budgets:", BUDGETS, "| output ->", OUT_DIR)

train planes: [450, 510, 570] | valid: 600 | horizon: 30
budgets: {'buy_again': 1000000000, 'popularity': 50, 'als_new': 50} | output -> C:\Users\siava\Recommendation_system\retail-recsys-platform\data\processed


In [3]:
def build_plane(as_of: int) -> pd.DataFrame:
    """Everything known at `as_of` for each pooled candidate, plus the 30-day label."""
    print(f"\n=== plane {as_of} ===")

    # --- ALS refit on this plane only (leakage rule 1) ---
    counts = q(f"""
        SELECT household_key, product_id, COUNT(DISTINCT basket_id) AS times_bought
        FROM staging.stg_transactions WHERE day_no <= {as_of}
        GROUP BY household_key, product_id
    """)
    households = np.sort(counts["household_key"].unique())
    products   = np.sort(counts["product_id"].unique())
    hh_pos = {h: i for i, h in enumerate(households)}
    pr_pos = {p: i for i, p in enumerate(products)}

    matrix = sp.csr_matrix(
        (np.log1p(counts["times_bought"]).astype(np.float32),
         (counts["household_key"].map(hh_pos), counts["product_id"].map(pr_pos))),
        shape=(len(households), len(products)))

    als = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=20.0,
                                  iterations=20, random_state=42)
    als.fit(matrix)

    # --- candidates: three sources ---
    buy_again = q(f"""
        SELECT household_key, product_id FROM household_product_snapshot({as_of})
    """).assign(source="buy_again")

    popularity = q(f"""
        WITH top_products AS (
            SELECT product_id FROM staging.stg_transactions
            WHERE day_no <= {as_of} AND day_no > {as_of} - 365
            GROUP BY product_id ORDER BY COUNT(*) DESC LIMIT {BUDGETS['popularity']}
        )
        SELECT h.household_key, t.product_id
        FROM (SELECT DISTINCT household_key FROM customer_snapshot({as_of})) h
        CROSS JOIN top_products t
    """).assign(source="popularity")

    n_als = BUDGETS["als_new"]
    ids, _ = als.recommend(np.arange(len(households)), matrix,
                           N=n_als, filter_already_liked_items=True)
    als_new = pd.DataFrame({
        "household_key": np.repeat(households, n_als),
        "product_id":    products[ids.ravel()],
        "source":        "als_new",
    })

    pool = pd.concat([buy_again, popularity, als_new], ignore_index=True)
    cand = (pool.pivot_table(index=["household_key", "product_id"],
                             columns="source", aggfunc="size", fill_value=0)
            .astype(bool).reset_index())
    for col in ["buy_again", "popularity", "als_new"]:          # guarantee all 3 flags exist
        if col not in cand:
            cand[col] = False
    cand = cand.rename(columns={"buy_again": "in_buy_again",
                                "popularity": "in_popularity",
                                "als_new": "in_als_new"})

    # --- features ---
    inter = q(f"""
        SELECT household_key, product_id, times_bought, days_since_last, due_ness
        FROM household_product_snapshot({as_of})
    """)
    cust = q(f"""
        SELECT household_key, days_since_last AS hh_days_since_last, tenure_days,
               baskets_30d, baskets_90d, baskets_365d,
               spend_30d, spend_365d, products_90d, avg_basket_value
        FROM customer_snapshot({as_of})
    """)
    item = q(f"""
        SELECT product_id,
               COUNT(DISTINCT basket_id)     AS item_baskets_365d,
               COUNT(DISTINCT household_key) AS item_households_365d,
               SUM(sales_value) / NULLIF(SUM(quantity), 0) AS item_avg_price
        FROM staging.stg_transactions
        WHERE day_no <= {as_of} AND day_no > {as_of} - 365
        GROUP BY product_id
    """)
    cycles = q("SELECT product_id, median_gap_days, gap_source FROM marts.mart_product_cycles")

    df = (cand
          .merge(inter,  on=["household_key", "product_id"], how="left")
          .merge(cust,   on="household_key",  how="left")
          .merge(item,   on="product_id",     how="left")
          .merge(cycles, on="product_id",     how="left"))

    # times_bought: absent means zero (a fact). days_since_last / due_ness stay NaN (informative).
    df["times_bought"] = df["times_bought"].fillna(0)

    # --- ALS affinity for every candidate pair ---
    u = df["household_key"].map(hh_pos).to_numpy()
    i = df["product_id"].map(pr_pos).to_numpy()
    ok = ~(pd.isna(u) | pd.isna(i))
    affinity = np.full(len(df), np.nan, dtype=np.float32)
    affinity[ok] = np.einsum("ij,ij->i",
                             als.user_factors[u[ok].astype(int)],
                             als.item_factors[i[ok].astype(int)])
    df["als_affinity"] = affinity

    # --- label: the future slice ---
    labels = q(f"""
        SELECT household_key, product_id, 1 AS label
        FROM purchase_labels({as_of}, {HORIZON})
    """)
    df = df.merge(labels, on=["household_key", "product_id"], how="left")
    df["label"] = df["label"].fillna(0).astype(np.int8)
    df["asof_day"] = as_of

    print(f"rows {len(df):,} | positives {df['label'].mean():.3%} | cols {df.shape[1]}")
    return df

In [4]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.concat([build_plane(d) for d in TRAIN_DAYS], ignore_index=True)
valid = build_plane(VALID_DAY)

train.to_parquet(OUT_DIR / "ranker_train.parquet", index=False)
valid.to_parquet(OUT_DIR / "ranker_valid.parquet", index=False)

print(f"\ntrain {train.shape} -> ranker_train.parquet")
print(f"valid {valid.shape} -> ranker_valid.parquet")
train.head()


=== plane 450 ===


100%|██████████| 20/20 [00:03<00:00,  5.08it/s]


rows 1,085,681 | positives 4.540% | cols 25

=== plane 510 ===


100%|██████████| 20/20 [00:04<00:00,  4.82it/s]


rows 1,197,974 | positives 4.239% | cols 25

=== plane 570 ===


100%|██████████| 20/20 [00:04<00:00,  4.09it/s]


rows 1,303,280 | positives 3.967% | cols 25

=== plane 600 ===


100%|██████████| 20/20 [00:06<00:00,  3.32it/s]


rows 1,358,184 | positives 3.838% | cols 25

train (3586935, 25) -> ranker_train.parquet
valid (1358184, 25) -> ranker_valid.parquet


,household_key,product_id,in_als_new,in_buy_again,in_popularity,times_bought,days_since_last,due_ness,hh_days_since_last,tenure_days,...,products_90d,avg_basket_value,item_baskets_365d,item_households_365d,item_avg_price,median_gap_days,gap_source,als_affinity,label,asof_day
0,1,821815,False,True,False,1.0,139.0,2.417391,14,399,...,169,54.421064,13.0,13.0,1.335333,57.5,sub_commodity,0.580728,0,450
1,1,822965,True,False,False,0.0,NaN,NaN,14,399,...,169,54.421064,89.0,67.0,1.217500,73.5,product,0.980012,0,450
2,1,823721,False,True,False,1.0,159.0,1.606061,14,399,...,169,54.421064,317.0,224.0,2.987962,99.0,product,0.839770,0,450
3,1,823990,False,True,False,1.0,304.0,3.415730,14,399,...,169,54.421064,365.0,252.0,4.714498,89.0,product,0.509662,0,450
4,1,825123,False,True,False,3.0,304.0,4.108108,14,399,...,169,54.421064,56.0,36.0,3.161833,74.0,sub_commodity,0.954202,0,450


In [7]:
zero_fill = ["spend_30d", "spend_365d", "item_baskets_365d", "item_households_365d"]
for df in (train, valid):
    df[zero_fill] = df[zero_fill].fillna(0)

train.to_parquet(OUT_DIR / "ranker_train.parquet", index=False)
valid.to_parquet(OUT_DIR / "ranker_valid.parquet", index=False)
print(train.isna().mean()[lambda s: s > 0].round(3))

days_since_last    0.139
due_ness           0.139
item_avg_price     0.014
dtype: float64


In [8]:
# 1. positives per plane — should be similar; a wobble means that period differs
print(train.groupby("asof_day")["label"].agg(["size", "mean"]).round(4), "\n")

# 2. missingness per column — expect NaN only in interaction features + cycles
missing = train.isna().mean().sort_values(ascending=False)
print(missing[missing > 0].round(3), "\n")

# 3. does each source actually carry signal? (label rate by flag)
for flag in ["in_buy_again", "in_popularity", "in_als_new"]:
    print(f"{flag:16s} n={train[flag].sum():>10,}  label rate={train.loc[train[flag], 'label'].mean():.3%}")

             size    mean
asof_day                 
450       1085681  0.0454
510       1197974  0.0424
570       1303280  0.0397 

due_ness           0.139
days_since_last    0.139
item_avg_price     0.014
dtype: float64 

in_buy_again     n= 3,088,900  label rate=4.665%
in_popularity    n=   374,700  label rate=7.302%
in_als_new       n=   374,700  label rate=1.567%


In [9]:
numeric = ["times_bought", "days_since_last", "due_ness", "als_affinity",
           "item_baskets_365d", "baskets_90d", "spend_365d", "median_gap_days"]

for col in numeric:
    bucket = pd.qcut(train[col], 10, duplicates="drop")
    rate = train.groupby(bucket, observed=True)["label"].mean()
    print(f"\n{col}: label rate from {rate.iloc[0]:.3%} (low) to {rate.iloc[-1]:.3%} (high)")


times_bought: label rate from 1.917% (low) to 25.387% (high)

days_since_last: label rate from 17.339% (low) to 1.008% (high)

due_ness: label rate from 14.541% (low) to 1.522% (high)

als_affinity: label rate from 0.920% (low) to 8.572% (high)

item_baskets_365d: label rate from 1.151% (low) to 7.439% (high)

baskets_90d: label rate from 1.404% (low) to 6.086% (high)

spend_365d: label rate from 1.613% (low) to 6.301% (high)

median_gap_days: label rate from 6.008% (low) to 2.324% (high)
